In [ ]:
"""
Testing data loading with this repo's setup (utils.py).
Making a cleaner way to test the model.
"""

import sys
from pathlib import Path
from art.attacks.evasion import FastGradientMethod
from art.estimators.classification import PyTorchClassifier


from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from sklearn.preprocessing import MinMaxScaler

import time
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[2]))

import sys
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

import os

import torch
from attacks.fl.models import OBU
from utils.functions import get_windowed_data, load_model_checkpoint

In [ ]:
# ----
# Input
model_filename = "randpos-30-10-100.ckpt"

# ----
# Load model
model = load_model_checkpoint(model_filename, gpu=False)


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [26]:
# ----
# Input
train_perc = 80
data_file = f"../../../data/RandomPos_0709.csv"

# ----
# Load data
(x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, train_perc)

In [27]:
# ----
# Test the model
out = model.test(x_test, y_test, mathy=True)
print(f"Out: {out}")

torch.Size([124774, 10, 20])
0.9152994345712776
0.9768519018073915
Model got 1205649/1247740 right.
Accuracy: 0.966266209306426, Precision: 0.9152994345712776, Recall: 0.9768519018073915, F1 Score: 0.9450745045535273
877040, 70.29028483498165% Zeroes, 370700 Non Zero entries.
Out: (0.9450745045535273, 0.9768519018073915, 0.9152994345712776, 0.966266209306426)


In [ ]:

class ARTCfCWrapper(nn.Module):
    def __init__(self, modena_model):
        super().__init__()
        self.modena_model = modena_model

    def forward(self, x):
        logits, _ = self.modena_model(x)

        # Shape:
        # (batch, seq_len, num_classes)
        return logits

class SequenceCrossEntropy(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.CrossEntropyLoss()

    def forward(self, a, b):
        # ART passes one-hot labels; CrossEntropyLoss needs class indices
        if b.dim() == 3:
            b = b.argmax(dim=-1)  # (batch, seq_len, num_classes) → (batch, seq_len)
        return self.loss(
            a.permute(0, 2, 1),  # (batch, seq_len, C) → (batch, C, seq_len)
            b.long()
        )
        
wrapped_model = ARTCfCWrapper(model.model)
criterion = SequenceCrossEntropy()
optimizer = optim.Adam(
    wrapped_model.parameters(),
    lr=0.001
)

classifier = PyTorchClassifier(
    model=wrapped_model,
    loss=criterion,
    optimizer=optimizer,
    input_shape=(10, 7),
    nb_classes=20,
    clip_values=None, # change this
)

print(type(model))
print(type(model.model))
print(type(wrapped_model))



<class 'attacks.fl.models.OBU'>
<class 'attacks.fl.models.Modena'>
<class '__main__.ARTCfCWrapper'>
Adversarial accuracy: 0.9683812332697517


In [33]:
# ----
# # Test the model
benign_predictions = classifier.predict(x_test)
pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
benign_accuracy = np.sum(pred_classes == y_test) / pred_classes.size
print(f"Benign accuracy: {benign_accuracy}")

Benign accuracy: 0.966266209306426


In [37]:
# Adversarial test
def run_fgsm (eps):
    attack = FastGradientMethod(estimator=classifier, eps=eps)
    x_test_adv = attack.generate(x=x_test)

    adversarial_predictions = classifier.predict(x_test_adv)
    pred_classes = np.argmax(adversarial_predictions, axis=-1)  # (N, seq_len)
    adversarial_accuracy = np.sum(pred_classes == y_test) / pred_classes.size
    print(f"Adversarial accuracy (eps {eps}): {adversarial_accuracy}")

In [42]:
for i in range(1, 20, 1):
    eps = float(i/100)
    run_fgsm(eps)

Adversarial accuracy (eps 0.01): 0.966503438216295
Adversarial accuracy (eps 0.02): 0.9668087902928495
Adversarial accuracy (eps 0.03): 0.96703880616154
Adversarial accuracy (eps 0.04): 0.967248785804735
Adversarial accuracy (eps 0.05): 0.9674643755910687
Adversarial accuracy (eps 0.06): 0.9676559219068075
Adversarial accuracy (eps 0.07): 0.9678570856107843
Adversarial accuracy (eps 0.08): 0.9680390145382852
Adversarial accuracy (eps 0.09): 0.9682249507108853
Adversarial accuracy (eps 0.1): 0.9683812332697517
Adversarial accuracy (eps 0.11): 0.9685351114815587
Adversarial accuracy (eps 0.12): 0.9686561302835527
Adversarial accuracy (eps 0.13): 0.9687787519835863
Adversarial accuracy (eps 0.14): 0.9689182041130364
Adversarial accuracy (eps 0.15): 0.9690183852405149
Adversarial accuracy (eps 0.16): 0.9691698591052623
Adversarial accuracy (eps 0.17): 0.9693085097856925
Adversarial accuracy (eps 0.18): 0.96941349960729
Adversarial accuracy (eps 0.19): 0.9695601647779185
